# STOR 120: Mastering `.group()` and `.pivot()`
### An Interactive Step-by-Step Lab

This lab is structured to guide you from basic single-column counting to advanced multi-variable cross-classification[cite: 4, 6].

#### Lab Workflow:
1. **Read the concept card and syntax explanation**[cite: 4, 6].
2. **Write your code** in the question cells by replacing `...`.
3. **Run the check cell** right beneath your answer to test your solution[cite: 1, 4].

---
## Environment Setup
Run this cell first to load the required course libraries[cite: 1, 9].

In [2]:
from datascience import *
import numpy as np
import matplotlib.pyplot as plots
%matplotlib inline
plots.style.use('fivethirtyeight')

print("✅ Setup cell completed! Course tools imported.")

✅ Setup cell completed! Course tools imported.


---
## Part 1: Single-Column Counting with `.group()`

### Concept Card
The simplest form of `.group()` takes a single column label[cite: 4, 6]. It finds every unique value in that column and counts how many rows have that value[cite: 4, 5, 6].

```python
table.group('category_column')
```

* **Categories:** Sorted alphabetically or in ascending order[cite: 3, 5].
* **Count column:** Always named `'count'` by default[cite: 3, 4, 5].

Let's load the UNC student sample table `unc_students`[cite: 7]:

In [3]:
unc_students = Table().with_columns(
    'Name', make_array('Alice', 'Henry', 'Nivedha', 'Melanie', 'Lily', 'Julia', 'Josh', 'Sara', 'Luca'),
    'Grade', make_array('Freshman', 'Senior', 'Sophomore', 'Junior', 'Sophomore', 'Sophomore', 'Junior', 'Sophomore', 'Junior'),
    'Major', make_array('Business', 'Biology', 'Business', 'English', 'Mathematics', 'English', 'Biology', 'English', 'Business'),
    'Credits', make_array(12, 14, 16, 13, 17, 18, 15, 13, 14)
)
unc_students

Name,Grade,Major,Credits
Alice,Freshman,Business,12
Henry,Senior,Biology,14
Nivedha,Sophomore,Business,16
Melanie,Junior,English,13
Lily,Sophomore,Mathematics,17
Julia,Sophomore,English,18
Josh,Junior,Biology,15
Sara,Sophomore,English,13
Luca,Junior,Business,14


### Exercise 1.1: Counting Categories
Find how many students belong to each `'Major'`. Assign the resulting table to `major_counts`.

In [ ]:
major_counts = unc_students.group("Major")
major_counts

Major,Name mean,Grade mean,Credits mean
Biology,,,14.5
Business,,,14
English,,,14.6667
Mathematics,,,17


In [5]:
# Exercise 1.1 Check Cell
assert isinstance(major_counts, Table), "major_counts must be a Table."
assert list(major_counts.labels) == ['Major', 'count'], "Labels should be ['Major', 'count']."
assert major_counts.where('Major', 'English').column('count').item(0) == 3, "There should be 3 English majors."
assert major_counts.num_rows == 4, "There are 4 unique majors."
print("✅ Exercise 1.1 passed!")

✅ Exercise 1.1 passed!


---
## Part 2: Single-Column Aggregation with `.group()`

### Concept Card
When you supply a second argument to `.group()`, it must be an aggregation **function name** (such as `np.mean`, `sum`, `min`, or `max`)[cite: 4, 6].

```python
table.group('category_column', function_name)
```

> **Midterm Watchout — Column Renaming:** `.group()` applies the function to **every remaining numerical column** and renames them by appending the function name[cite: 4, 8]. For example, `'Credits'` becomes `'Credits sum'` or `'Credits mean'`[cite: 4, 8].
>
> **Best Practice:** Use `.select()` before grouping to keep only the columns you care about[cite: 4, 8].

### Exercise 2.1: Aggregating by Group
Calculate the **average credits** taken by students in each `'Grade'`[cite: 4, 7].
1. Start by selecting only `'Grade'` and `'Credits'` from `unc_students`[cite: 4, 7, 8].
2. Group by `'Grade'` using `np.mean`[cite: 4, 7, 8].
3. Assign the resulting table to `avg_credits_by_grade`[cite: 4, 7].

In [7]:
avg_credits_by_grade = unc_students.select("Grade", "Credits").group("Grade", np.mean)
avg_credits_by_grade

Grade,Credits mean
Freshman,12
Junior,14
Senior,14
Sophomore,16


In [ ]:
# Exercise 2.1 Check Cell
assert isinstance(avg_credits_by_grade, Table), "avg_credits_by_grade must be a Table."
assert list(avg_credits_by_grade.labels) == ['Grade', 'Credits mean'], "Aggregated column label must be 'Credits mean'."
assert avg_credits_by_grade.where('Grade', 'Junior').column('Credits mean').item(0) == 14.0, "Junior average should be 14.0."
print("✅ Exercise 2.1 passed!")

---
## Part 3: Multi-Column Grouping

### Concept Card
To group by multiple categories at the same time, pass a **list or array of column labels** as the first argument[cite: 4, 6]:

```python
# Counts each unique combination of Category1 and Category2:
table.group(make_array('Category1', 'Category2'))
# or using standard brackets:
table.group(['Category1', 'Category2'])
```

The output contains one row for every **unique combination** of values present in the data[cite: 4, 6].

### Exercise 3.1: Multi-Column Frequency
Count how many students belong to each combination of `'Grade'` and `'Major'`. Assign the table to `grade_major_pairs`[cite: 4, 7].

In [12]:
grade_major_pairs = unc_students.pivot("Major", "Grade")
grade_major_pairs

Grade,Biology,Business,English,Mathematics
Freshman,0,1,0,0
Junior,1,1,1,0
Senior,1,0,0,0
Sophomore,0,1,2,1


In [ ]:
# Exercise 3.1 Check Cell
assert isinstance(grade_major_pairs, Table), "Must be a Table."
assert grade_major_pairs.num_columns == 3, "Table must have 3 columns: Grade, Major, and count."
assert 'count' in grade_major_pairs.labels, "Table must contain 'count' column."
soph_english = grade_major_pairs.where('Grade', 'Sophomore').where('Major', 'English').column('count').item(0)
assert soph_english == 2, "There should be 2 Sophomore English majors."
print("✅ Exercise 3.1 passed!")

---
## Part 4: Basic Contingency Grids with `.pivot()`

### Concept Card
While `.group(['Col1', 'Col2'])` displays combinations in tall, vertical rows, `.pivot()` cross-classifies data into a **two-dimensional grid**[cite: 4, 6, 8]:

```python
table.pivot(columns_label, rows_label)
```

| Argument Position | Keyword | Role in Output Grid |[cite: 4, 6]
| :--- | :--- | :--- |[cite: 4, 6]
| **1st Argument** | `columns` | Unique values become the **new column headers**[cite: 4, 6]
| **2nd Argument** | `rows` | Unique values become the **row categories**[cite: 4, 6]

> **Advantage of `.pivot()`:** It shows combinations that had **0 counts**, whereas `.group()` simply omits zero-count pairs[cite: 4].

### Exercise 4.1: Building a Frequency Grid
Construct a contingency table using `unc_students` where:
* `'Grade'` values form the **column headers**[cite: 4, 6, 7]
* `'Major'` values form the **rows**[cite: 4, 6, 7]

Assign this table to `student_grid`[cite: 4, 7].

In [17]:
student_grid = unc_students.pivot("Grade", "Major")
student_grid

Major,Freshman,Junior,Senior,Sophomore
Biology,0,1,1,0
Business,1,1,0,1
English,0,1,0,2
Mathematics,0,0,0,1


In [18]:
# Exercise 4.1 Check Cell
assert isinstance(student_grid, Table), "student_grid must be a Table."
assert student_grid.labels[0] == 'Major', "The row column (first column) should be Major."
assert 'Freshman' in student_grid.labels and 'Senior' in student_grid.labels, "Grades must be the column headers."
assert student_grid.num_rows == 4, "There should be 4 rows for the 4 majors."
print("✅ Exercise 4.1 passed!")

✅ Exercise 4.1 passed!


---
## Part 5: Aggregated Pivot Tables (`values` & `collect`)

### Concept Card
By default, `.pivot()` fills cells with row counts[cite: 4, 6]. To calculate numerical statistics across the grid, supply the two optional keyword arguments[cite: 4, 6]:

```python
table.pivot(columns_label, rows_label, values='column_to_aggregate', collect=function_name)
```

* `values`: The string name of the numerical column to summarize[cite: 4, 6].
* `collect`: The aggregation function (e.g. `sum`, `np.mean`, `max`)[cite: 4, 6].

> **Common Trap:** In the `datascience` library, the keywords are `columns` and `rows` (not `pivot_columns`)[cite: 6]. Never pass the output variable name into `collect` (e.g., `collect=np.mean`, NOT `collect=my_table`)[cite: 2, 4].

### Exercise 5.1: Aggregating Inside a Grid
Using `unc_students`, construct a pivot table showing the **total credits** (`sum`) taken for each `'Major'` across each `'Grade'`[cite: 4, 6, 7]:
* Column headers: `'Grade'`[cite: 4, 6, 7]
* Row labels: `'Major'`[cite: 4, 6, 7]
* Cell values: Sum of `'Credits'`[cite: 4, 6, 7]

Assign the result to `credits_pivot`[cite: 4, 7].

In [26]:
credits_pivot = unc_students.pivot(
    "Grade",
    "Major",
    values="Credits",
    collect=np.sum
    )
credits_pivot

Major,Freshman,Junior,Senior,Sophomore
Biology,0,15,14,0
Business,12,14,0,16
English,0,13,0,31
Mathematics,0,0,0,17


In [27]:
# Exercise 5.1 Check Cell
assert isinstance(credits_pivot, Table), "credits_pivot must be a Table."
assert credits_pivot.labels[0] == 'Major', "The first column should be 'Major'."
# Sophomore English: Julia (18) + Sara (13) = 31 credits
soph_eng_credits = credits_pivot.where('Major', 'English').column('Sophomore').item(0)
assert soph_eng_credits == 31, f"Expected 31 credits for Sophomore English majors, got {soph_eng_credits}."
print("✅ Exercise 5.1 passed!")

✅ Exercise 5.1 passed!


---
## Part 6: Comprehensive Comparison Challenge

Run the cell below to initialize `ice_cream`, a dataset tracking ice cream purchases[cite: 4]:

In [28]:
ice_cream = Table().with_columns(
    'Flavor', make_array('Chocolate', 'Strawberry', 'Chocolate', 'Strawberry', 'Chocolate', 'Vanilla', 'Vanilla'),
    'Size', make_array('Small', 'Small', 'Large', 'Large', 'Large', 'Small', 'Large'),
    'Price', make_array(3.50, 4.00, 5.50, 6.00, 5.50, 3.25, 5.00)
)
ice_cream

Flavor,Size,Price
Chocolate,Small,3.5
Strawberry,Small,4
Chocolate,Large,5.5
Strawberry,Large,6
Chocolate,Large,5.5
Vanilla,Small,3.25
Vanilla,Large,5


### Exercise 6.1: Side-by-Side Comparison
Answer both tasks below using `ice_cream`[cite: 4]:

1. **Task A (`.group`):** Group by `'Flavor'` to find the **maximum** price of each flavor[cite: 4]. Retain only `'Flavor'` and `'Price'` before grouping[cite: 4, 8]. Assign to `max_price_by_flavor`[cite: 4].
2. **Task B (`.pivot`):** Create a pivot table displaying the **average** (`np.mean`) price for every combination of `'Size'` (columns) and `'Flavor'` (rows)[cite: 4, 6]. Assign to `price_grid`[cite: 4].

In [33]:
# Task A: Grouping for maximum price
max_price_by_flavor = ice_cream.select('Flavor', 'Price').group("Flavor", np.max)

# Task B: Pivoting for average price
price_grid = ice_cream.pivot(
    "Size",
    "Flavor",
    values="Price",
    collect=np.mean
)

max_price_by_flavor, price_grid

(Flavor     | Price max
 Chocolate  | 5.5
 Strawberry | 6
 Vanilla    | 5,
 Flavor     | Large | Small
 Chocolate  | 5.5   | 3.5
 Strawberry | 6     | 4
 Vanilla    | 5     | 3.25)

In [34]:
# Exercise 6.1 Check Cell
assert list(max_price_by_flavor.labels) == ['Flavor', 'Price max'], "Task A column should be labeled 'Price max'."
assert max_price_by_flavor.where('Flavor', 'Chocolate').column('Price max').item(0) == 5.50, "Max chocolate price is 5.50."

assert 'Small' in price_grid.labels and 'Large' in price_grid.labels, "Columns in price_grid should be Small and Large."
assert price_grid.labels[0] == 'Flavor', "First column of price_grid should be Flavor."
strawberry_large = price_grid.where('Flavor', 'Strawberry').column('Large').item(0)
assert np.isclose(strawberry_large, 6.00), "Strawberry Large price should be 6.00."
print("✅ All exercises passed! You have mastered .group() and .pivot()!")

✅ All exercises passed! You have mastered .group() and .pivot()!


---
## Quick Reference Summary Sheet

| Operation | Syntax | Output Structure |[cite: 4, 6]
| :--- | :--- | :--- |[cite: 4, 6]
| **Count category frequencies** | `t.group('Category')` | 2 columns: `'Category'`, `'count'`[cite: 4, 6]
| **Aggregate across a category** | `t.select(...).group('Category', func)` | Appends func name to labels (`'Col func'`)[cite: 4, 6, 8]
| **Count paired categories** | `t.group(['Cat1', 'Cat2'])` | 1 row per unique pair in data[cite: 4, 6]
| **Cross-classify counts into grid** | `t.pivot('Col_Cat', 'Row_Cat')` | 2D contingency table with headers & row labels[cite: 4, 6]
| **Cross-classify aggregated values** | `t.pivot('Col_Cat', 'Row_Cat', values='Val', collect=func)` | 2D grid containing summarized statistic in each cell[cite: 4, 6]

In [38]:
type(ice_cream.take(3))

datascience.tables.Table